In [0]:
dbutils.widgets.text(
    "catalog_name",
    "workspace",
)

dbutils.widgets.text(
    "schema_name",
    "de_learning",
)

dbutils.widgets.text(
    "max_quarantine_ratio",
    "0.50",
)

dbutils.widgets.text(
    "job_run_id",
    "interactive",
)

In [0]:
from decimal import Decimal

from pyspark.sql import functions as F


catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")

max_quarantine_ratio = Decimal(
    dbutils.widgets.get("max_quarantine_ratio")
)

job_run_id = dbutils.widgets.get("job_run_id")

if not Decimal("0") <= max_quarantine_ratio <= Decimal("1"):
    raise ValueError(
        "max_quarantine_ratio must be between 0 and 1."
    )

namespace = f"`{catalog_name}`.`{schema_name}`"

tables = {
    "bronze": f"{namespace}.`lf_bronze_sales`",
    "silver": f"{namespace}.`lf_silver_sales`",
    "quarantine": f"{namespace}.`lf_quarantine_sales`",
    "gold": f"{namespace}.`lf_gold_product_sales`",
}

print(f"Job run: {job_run_id}")
print(f"Checking namespace: {namespace}")

Job run: interactive
Checking namespace: `workspace`.`de_learning`


In [0]:
counts = {
    name: spark.table(table_name).count()
    for name, table_name in tables.items()
}

for name, row_count in counts.items():
    print(f"{name}: {row_count}")

if counts["bronze"] == 0:
    raise RuntimeError(
        "Bronze is empty; this exercise expects ingested data."
    )

if counts["bronze"] != (
    counts["silver"] + counts["quarantine"]
):
    raise RuntimeError(
        "Routing check failed: Bronze count does not equal "
        "Silver count plus Quarantine count."
    )

quarantine_ratio = (
    Decimal(counts["quarantine"])
    / Decimal(counts["bronze"])
)

print(f"Quarantine ratio: {quarantine_ratio:.1%}")
print(f"Allowed maximum: {max_quarantine_ratio:.1%}")

if quarantine_ratio > max_quarantine_ratio:
    raise RuntimeError(
        f"Quarantine ratio {quarantine_ratio:.1%} exceeds "
        f"the allowed maximum {max_quarantine_ratio:.1%}."
    )

silver_total = (
    spark.table(tables["silver"])
    .agg(F.sum("total_amount").alias("total"))
    .first()["total"]
)

gold_total = (
    spark.table(tables["gold"])
    .agg(F.sum("revenue").alias("total"))
    .first()["total"]
)

silver_total = silver_total or Decimal("0")
gold_total = gold_total or Decimal("0")

print(f"Silver total: {silver_total}")
print(f"Gold total:   {gold_total}")

if silver_total != gold_total:
    raise RuntimeError(
        "Revenue reconciliation failed: "
        "Gold revenue differs from Silver total_amount."
    )

print("All output checks passed.")

bronze: 6
silver: 4
quarantine: 2
gold: 4
Quarantine ratio: 33.3%
Allowed maximum: 50.0%
Silver total: 5840.00
Gold total:   5840.00
All output checks passed.
